# Geometry-MMALS G1 v1.0.9
## Stationary Route Geometry Qualification Pilot

**Scientific status:** executable five-seed pilot protocol.  
**Accepted claim before execution:** C0 implementation only.  
**Not claimed:** final C1 qualification, predictive superiority, host specialization, memory transport, operational utility, backward transfer, replay-free continual learning, or quantum advantage.

v1.0.9 is a focused follow-up to the executed v1.0.8 study. The strongest v1.0.8 result was a substantial d4 improvement in context order, route order, fiber consistency, held-out factor decoding, and source-disjoint causal specificity. The principal remaining protocol defect was that the legacy route target depended on the largest factor gap visible at the current continual stage. The same angle pair therefore received contradictory targets as the curriculum expanded.

This notebook freezes the factor topology and asks whether the d4 result replicates across five model seeds.


## Research history

| Version | Main correction | Outcome motivating the next step |
|---|---|---|
| v1.0.0-v1.0.2 | Falsifiable geometry specification and numerical stability | C0 scaffold and finite optimization |
| v1.0.3 | Same-source cross-angle training unit | First informative local geometry run |
| v1.0.4 | Matched initialization and compute | Geometry separated from extra forwards |
| v1.0.5-v1.0.6 | Context mediation, static routes, counterbalanced curricula | Direct sensory shortcut and curriculum sensitivity identified |
| v1.0.7 | Forced context bottleneck | Context shown sufficient and causally informative when the shortcut is removed |
| v1.0.8 | Direct context geometry and cross-source alignment | Strong single-seed d4 candidate geometry; predictive benefit still small; route target found nonstationary |
| **v1.0.9** | **Stationary route target and five-seed pilot** | Implemented here |


## Tracked changes in v1.0.9

- **CHG-109-01** - Add `paired_route_geometry_loss_stationary` with a fixed 120-degree factor span.
- **CHG-109-02** - Use the same chord-compatible target for route and normalized context geometry.
- **CHG-109-03** - Retain a legacy-route full-alignment comparator to isolate the target correction.
- **CHG-109-04** - Restrict the primary architecture to context dimension 4.
- **CHG-109-05** - Add five fixed-split model seeds for the pilot profile.
- **CHG-109-06** - Add a dense 15-degree evaluation grid from -75 to +75 degrees.
- **CHG-109-07** - Export paired source-bootstrap prediction intervals by held-out angle.
- **CHG-109-08** - Export source-bootstrap causal-specificity intervals.
- **CHG-109-09** - Add class-log-probability and prediction-identity preservation under causal interventions.
- **CHG-109-10** - Aggregate method and treatment effects with model seed as the statistical unit.
- **CHG-109-11** - Keep the v1.0.8 source split and sensory boundary fixed for direct comparability.
- **CHG-109-12** - Separate pilot candidate gates from final qualification claims.
- **CHG-109-13** - Archive the complete v1.0.8 result bundle and results report.
- **CHG-109-14** - Preserve the reviewer Status and Perspective report v1.1 and LaTeX sources.


## Mathematical correction

For a route distribution $r\in\Delta^{H-1}$, use square-root simplex coordinates $q=\sqrt r$. The normalized chord distance between two routes is

\[
d_R(r_a,r_b)=\sqrt{1-\langle q_a,q_b\rangle}\in[0,1].
\]

The fixed factor target is

\[
\tau(u_a,u_b)=\sin\left[\frac{\pi}{2}\min\left(\frac{|u_a-u_b|}{120^\circ},1\right)\right].
\]

The stationary route loss is

\[
\mathcal L_{R,\mathrm{stat}}=
\mathbb E_{s,a<b}\left(d_R(r_{s,a},r_{s,b})-\tau(u_a,u_b)\right)^2
+\lambda_{R,\mathrm{far}}\,\mathcal L_{R,\mathrm{far}}.
\]

Unlike the v1.0.8 legacy target, $\tau(-60^\circ,-30^\circ)$ is identical whether two or five training angles have already been observed.


## 0. Setup and release-integrity gate


In [ ]:
import os, sys, shutil, importlib, subprocess
from pathlib import Path

EXPECTED_PACKAGE_VERSION = "1.0.9"
REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
FORCE_REFRESH = False

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
for name in list(sys.modules):
    if name == "geometry_mmalls" or name.startswith("geometry_mmalls."):
        del sys.modules[name]
importlib.invalidate_caches()
import geometry_mmalls

loaded_from = Path(geometry_mmalls.__file__).resolve()
assert loaded_from.parent == (SRC_DIR / "geometry_mmalls").resolve()
assert geometry_mmalls.__version__ == EXPECTED_PACKAGE_VERSION, (
    f"Expected {EXPECTED_PACKAGE_VERSION}, loaded {geometry_mmalls.__version__}. "
    "Push/pull v1.0.9 and rerun with FORCE_REFRESH=True."
)
print("Geometry-MMALS", geometry_mmalls.__version__, "from", loaded_from)


## 1. Configuration, profiles, and imports


In [ ]:
from pathlib import Path
import copy, hashlib, json, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from scipy import stats
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import (
    paired_route_geometry_loss,
    paired_route_geometry_loss_stationary,
    paired_context_geometry_loss,
    context_path_spread_loss,
    cross_source_fiber_alignment_loss,
    factor_centroid_geometry_loss,
)
from geometry_mmalls.metrics import (
    bootstrap_mean_ci,
    grouped_geometry_scores,
    centroid_geometry_scores,
    fiber_alignment_scores,
    ridge_factor_probe,
    context_collapse_diagnostics,
)
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder

NOTEBOOK_VERSION = "1.0.9"
BUILD_REVISION = "stationary-route-qualification-pilot-r1"
base_config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())
protocol = yaml.safe_load(Path("configs/rotated_mnist_g1_v109.yaml").read_text())
assert protocol["build_revision"] == BUILD_REVISION

# Profiles: development=one seed, pilot=five model seeds, qualification=ten model seeds.
RUN_PROFILE = "pilot"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_SEED = int(protocol["fixed_split_seed"])
SENSORY_SEED = int(protocol["fixed_sensory_seed"])
if RUN_PROFILE == "development":
    MODEL_SEEDS = [0]
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 512, 256
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 1000
    SENSORY_EPOCHS, STAGE_EPOCHS, SOURCE_BATCH_SIZE = 2, 2, 64
    SOURCE_BOOTSTRAP_SAMPLES = 500
    SENSORY_GATE = 0.85
elif RUN_PROFILE == "pilot":
    MODEL_SEEDS = list(map(int, protocol["pilot_model_seeds"]))
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 512, 256
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 1000
    SENSORY_EPOCHS, STAGE_EPOCHS, SOURCE_BATCH_SIZE = 2, 2, 64
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
    SENSORY_GATE = 0.85
elif RUN_PROFILE == "qualification":
    MODEL_SEEDS = list(map(int, protocol["qualification_model_seeds"]))
    TRAIN_SOURCE_LIMIT = int(base_config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT, SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 12000, 5000
    SENSORY_EPOCHS = int(base_config["training"]["sensory_pretrain_epochs"])
    STAGE_EPOCHS = int(base_config["training"]["stage_epochs"])
    SOURCE_BATCH_SIZE = int(base_config["paired_protocol"]["source_batch_size"])
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
    SENSORY_GATE = 0.95
else:
    raise ValueError("RUN_PROFILE must be development, pilot, or qualification")

train_angles = list(map(float, base_config["data"]["train_angles"]))
PRIMARY_CURRICULUM = list(map(float, protocol["primary_curriculum"]))
DENSE_EVAL_ANGLES = list(map(float, protocol["dense_eval_angles"]))
DENSE_HELDOUT_ANGLES = [a for a in DENSE_EVAL_ANGLES if a not in train_angles]
CONTEXT_DIM = int(protocol["context_dimension"])
W = protocol["loss_weights"]
CG = protocol["context_geometry"]
RG = protocol["route_geometry"]
FA = protocol["fiber_alignment"]

METHOD_SPECS = {
    "d4_no_geo": {"route_mode": "none", "full_alignment": False},
    "d4_full_legacy_route": {"route_mode": "legacy", "full_alignment": True},
    "d4_full_stationary_route": {"route_mode": "stationary", "full_alignment": True},
}
METHODS = list(METHOD_SPECS)
print({
    "version": NOTEBOOK_VERSION,
    "revision": BUILD_REVISION,
    "profile": RUN_PROFILE,
    "device": str(DEVICE),
    "model_seeds": MODEL_SEEDS,
    "methods": METHODS,
})


## 2. Stationarity and numerical self-tests


In [ ]:
factors_two = torch.tensor([-60.0, -30.0])
factors_full = torch.tensor([-60.0, -30.0, 0.0, 30.0, 60.0])
logits_two = torch.randn(8, 2, 4, requires_grad=True)
routes_two = torch.softmax(logits_two, dim=-1)
loss_two, diag_two = paired_route_geometry_loss_stationary(
    routes_two,
    factors_two,
    max_factor_span=RG["max_factor_span_degrees"],
    far_threshold_degrees=RG["far_threshold_degrees"],
    far_route_margin=RG["far_route_margin"],
    far_weight=RG["far_weight"],
    match_weight=RG["match_weight"],
    return_diagnostics=True,
)
loss_two.backward()
assert torch.isfinite(loss_two)
assert logits_two.grad is not None and torch.isfinite(logits_two.grad).all()

# The helper target is fixed by the declared 120-degree topology, not the stage span.
from geometry_mmalls.geometry import context_chord_target
expected_gap_30 = context_chord_target(torch.tensor(30.0), 120.0)
assert torch.allclose(expected_gap_30, torch.sin(torch.tensor(math.pi / 8.0)), atol=1e-7)

uniform = torch.full((8, 5, 4), 0.25)
assert paired_route_geometry_loss_stationary(uniform, factors_full) > 0.05
print("Stationary route target, collapsed-route penalty, and backward gates: PASS")


## 3. Fixed source split and common frozen sensory grove


In [ ]:
root = Path(base_config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)

split_rng = np.random.default_rng(SPLIT_SEED)
train_perm = split_rng.permutation(len(base_train))
test_perm = split_rng.permutation(len(base_test))
sensory_indices = train_perm[:SENSORY_SOURCE_LIMIT].tolist()
train_indices = train_perm[:TRAIN_SOURCE_LIMIT].tolist()
test_indices = test_perm[:TEST_SOURCE_LIMIT].tolist()
sensory_test_indices = test_perm[:SENSORY_TEST_LIMIT].tolist()
half = len(test_indices) // 2
tangent_fit_indices = test_indices[:half]
tangent_eval_indices = test_indices[half:]

def checksum(values):
    return hashlib.sha256(",".join(map(str, values)).encode()).hexdigest()

def generator(seed):
    g = torch.Generator(); g.manual_seed(int(seed)); return g

def fixed_loader(angle, train, indices, shuffle, loader_seed=0, batch_size=128):
    ds = Subset(FixedAngleMNIST(root, angle=angle, train=train, download=True), indices)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,
        generator=generator(loader_seed) if shuffle else None,
    )

def multi_loader(angles, train, indices, shuffle, loader_seed=0):
    ds = MultiAngleMNIST(root, angles=angles, train=train, indices=indices, download=True)
    return DataLoader(
        ds, batch_size=SOURCE_BATCH_SIZE, shuffle=shuffle, num_workers=0,
        generator=generator(loader_seed) if shuffle else None,
    )

split_manifest = {
    "split_seed": SPLIT_SEED,
    "train_sha256": checksum(train_indices),
    "test_sha256": checksum(test_indices),
    "tangent_fit_sha256": checksum(tangent_fit_indices),
    "tangent_eval_sha256": checksum(tangent_eval_indices),
}

random.seed(SENSORY_SEED); np.random.seed(SENSORY_SEED); torch.manual_seed(SENSORY_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SENSORY_SEED)
latent_dim = int(base_config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
sensory_optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(sensory_head.parameters()),
    lr=float(base_config["training"]["learning_rate"]),
    weight_decay=float(base_config["training"]["weight_decay"]),
)
sensory_history = []
for epoch in range(SENSORY_EPOCHS):
    total = correct = 0; loss_sum = 0.0
    encoder.train(); sensory_head.train()
    for x, y, _, _ in fixed_loader(0.0, True, sensory_indices, True, SENSORY_SEED * 1000 + epoch):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x)); loss = F.cross_entropy(logits, y)
        sensory_optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(sensory_head.parameters()), 5.0)
        sensory_optimizer.step()
        total += y.numel(); correct += (logits.argmax(1) == y).sum().item(); loss_sum += float(loss.detach()) * y.numel()
    sensory_history.append({"epoch": epoch, "loss": loss_sum / total, "train_accuracy": correct / total})
    print(sensory_history[-1])

@torch.no_grad()
def evaluate_sensory():
    encoder.eval(); sensory_head.eval(); total = correct = 0
    for x, y, _, _ in fixed_loader(0.0, False, sensory_test_indices, False, batch_size=256):
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (sensory_head(encoder(x)).argmax(1) == y).sum().item(); total += y.numel()
    return correct / total

sensory_accuracy = evaluate_sensory()
assert RUN_PROFILE != "qualification" or sensory_accuracy >= SENSORY_GATE
sensory_state = copy.deepcopy(encoder.state_dict())
print("Sensory gate:", sensory_accuracy, "threshold:", SENSORY_GATE)


## 4. Model, training, and equal-compute helpers


In [ ]:
def build_model():
    enc = SmallConvEncoder(latent_dim).to(DEVICE); enc.load_state_dict(sensory_state)
    return GeometryMMALS(
        enc,
        latent_dim=latent_dim,
        context_dim=CONTEXT_DIM,
        num_hosts=int(base_config["model"]["num_hosts"]),
        host_hidden_dim=int(base_config["model"]["host_hidden_dim"]),
        freeze_encoder=True,
        bottleneck_hidden_dim=64,
    ).to(DEVICE)

def state_hash(state_dict):
    h = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        h.update(name.encode()); h.update(tensor.numpy().tobytes())
    return h.hexdigest()

def forward_mode(model, x):
    z0 = model.encode(x)
    context_raw, context = model.infer_context(z0)
    route = torch.softmax(model.context_bottleneck_router(context), dim=-1)
    return model.synthesize(z0, context, route, context_raw=context_raw)

def host_diversity(host_outputs):
    h = F.normalize(host_outputs, dim=-1)
    similarity = torch.einsum("bhd,bjd->bhj", h, h)
    mask = ~torch.eye(similarity.shape[-1], dtype=torch.bool, device=similarity.device)
    return similarity[:, mask].square().mean()

def zero_like(tensor):
    return tensor.sum() * 0.0

def epoch_seed(model_seed, stage, epoch):
    return int(model_seed) * 1_000_000 + stage * 10_000 + epoch

@torch.no_grad()
def evaluate_model(model, method, model_seed, stage, angles):
    rows = []; model.eval()
    for angle in angles:
        total = correct = 0; nll_sum = 0.0
        for x, y, _, _ in fixed_loader(angle, False, test_indices, False, batch_size=256):
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = forward_mode(model, x)
            total += y.numel(); correct += (trace.logits.argmax(1) == y).sum().item()
            nll_sum += float(F.cross_entropy(trace.logits, y, reduction="sum"))
        rows.append({
            "model_seed": model_seed, "method": method, "stage": stage,
            "angle": float(angle),
            "angle_type": "train" if angle in train_angles else "heldout",
            "accuracy": correct / total, "nll": nll_sum / total,
        })
    return rows

def route_loss_for_mode(routes, factors, route_mode):
    if route_mode == "none":
        zero = zero_like(routes)
        return zero, {"route_match": zero, "route_far": zero}
    if route_mode == "legacy":
        value = paired_route_geometry_loss(routes, factors)
        return value, {"route_match": value, "route_far": zero_like(value)}
    if route_mode == "stationary":
        return paired_route_geometry_loss_stationary(
            routes, factors,
            max_factor_span=RG["max_factor_span_degrees"],
            far_threshold_degrees=RG["far_threshold_degrees"],
            far_route_margin=RG["far_route_margin"],
            far_weight=RG["far_weight"],
            match_weight=RG["match_weight"],
            return_diagnostics=True,
        )
    raise ValueError(f"Unknown route mode: {route_mode}")

def train_method(model_seed, method, spec, initial_state):
    model = build_model(); model.load_state_dict(initial_state, strict=True)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params,
        lr=float(base_config["training"]["learning_rate"]),
        weight_decay=float(base_config["training"]["weight_decay"]),
    )
    stage_rows = []; evaluation_rows = []
    for stage, current_angle in enumerate(PRIMARY_CURRICULUM):
        seen = PRIMARY_CURRICULUM[: stage + 1]
        keys = [
            "loss", "current_ce", "anchor_ce", "route_geo", "route_match", "route_far",
            "context_geo", "context_spread", "fiber", "centroid", "host_diversity",
        ]
        totals = {key: 0.0 for key in keys}
        totals.update({"source_examples": 0, "forward_images": 0, "optimizer_steps": 0})
        start = time.perf_counter(); model.train()
        for epoch in range(STAGE_EPOCHS):
            loader = multi_loader(seen, True, train_indices, True, epoch_seed(model_seed, stage, epoch))
            for views, y, factors, _ in loader:
                batch, angle_count = views.shape[:2]
                flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
                y, factors = y.to(DEVICE), factors.to(DEVICE)
                trace = forward_mode(model, flat)
                logits = trace.logits.reshape(batch, angle_count, -1)
                routes = trace.route.reshape(batch, angle_count, -1)
                contexts = trace.context.reshape(batch, angle_count, -1)
                hosts = trace.host_outputs.reshape(batch, angle_count, trace.host_outputs.shape[1], -1)

                current_ce = F.cross_entropy(logits[:, -1], y)
                anchor_ce = (
                    F.cross_entropy(
                        logits[:, :-1].reshape(-1, logits.shape[-1]),
                        y[:, None].expand(-1, angle_count - 1).reshape(-1),
                    ) if angle_count > 1 else zero_like(logits)
                )
                route_geo, route_diag = route_loss_for_mode(routes, factors, spec["route_mode"])
                if spec["full_alignment"]:
                    context_geo = paired_context_geometry_loss(
                        contexts, factors,
                        max_factor_span=CG["max_factor_span_degrees"],
                        far_threshold_degrees=CG["far_threshold_degrees"],
                        far_context_margin=CG["far_context_margin"],
                        far_weight=CG["far_weight"],
                    )
                    spread = context_path_spread_loss(contexts, CG["spread_floor"])
                    fiber = cross_source_fiber_alignment_loss(
                        contexts, factors,
                        direction_weight=FA["direction_weight"],
                        length_weight=FA["length_weight"],
                    )
                    centroid = factor_centroid_geometry_loss(
                        contexts, factors,
                        max_factor_span=CG["max_factor_span_degrees"],
                        far_threshold_degrees=CG["far_threshold_degrees"],
                        far_context_margin=CG["far_context_margin"],
                        far_weight=CG["far_weight"],
                    )
                else:
                    context_geo = spread = fiber = centroid = zero_like(contexts)
                diversity = host_diversity(hosts[:, -1])
                loss = (
                    current_ce
                    + float(W["retention_anchor"]) * anchor_ce
                    + float(W["route_geometry"]) * route_geo
                    + float(W["context_geometry"]) * context_geo
                    + float(W["context_spread"]) * spread
                    + float(W["fiber_alignment"]) * fiber
                    + float(W["centroid_alignment"]) * centroid
                    + float(base_config["losses"]["host_functional_diversity"]) * diversity
                )
                if not torch.isfinite(torch.stack([loss, current_ce, anchor_ce, route_geo, context_geo, spread, fiber, centroid, diversity])).all():
                    raise FloatingPointError(f"Non-finite loss for seed={model_seed}, method={method}, stage={stage}")
                optimizer.zero_grad(set_to_none=True); loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(params, 5.0)
                if not torch.isfinite(torch.as_tensor(grad_norm)):
                    raise FloatingPointError("Non-finite gradient norm")
                optimizer.step()

                totals["source_examples"] += batch
                totals["forward_images"] += batch * angle_count
                totals["optimizer_steps"] += 1
                values = {
                    "loss": loss, "current_ce": current_ce, "anchor_ce": anchor_ce,
                    "route_geo": route_geo, "route_match": route_diag["route_match"],
                    "route_far": route_diag["route_far"], "context_geo": context_geo,
                    "context_spread": spread, "fiber": fiber, "centroid": centroid,
                    "host_diversity": diversity,
                }
                for key, value in values.items(): totals[key] += float(value.detach()) * batch
        count = max(totals["source_examples"], 1)
        row = {
            "model_seed": model_seed, "method": method, "stage": stage,
            "current_angle": current_angle, "seen_angles": json.dumps(seen),
            "forward_images": totals["forward_images"], "optimizer_steps": totals["optimizer_steps"],
            "source_examples": totals["source_examples"], "wall_seconds": time.perf_counter() - start,
            **{key: totals[key] / count for key in keys},
        }
        stage_rows.append(row); print(row)
        evaluation_rows.extend(evaluate_model(model, method, model_seed, stage, DENSE_EVAL_ANGLES))
    return model, pd.DataFrame(stage_rows), pd.DataFrame(evaluation_rows)


## 5. Trace, geometry, prediction, and causal helpers


In [ ]:
@torch.no_grad()
def collect_trace(model, method, model_seed, angles, indices):
    rows = []; model.eval()
    for views, labels, factors, source_ids in multi_loader(angles, False, indices, False):
        batch, angle_count = views.shape[:2]
        trace = forward_mode(model, views.reshape(-1, *views.shape[2:]).to(DEVICE))
        logits = trace.logits.reshape(batch, angle_count, -1)
        arrays = {
            "context": trace.context.reshape(batch, angle_count, -1).cpu().numpy(),
            "context_raw": trace.context_raw.reshape(batch, angle_count, -1).cpu().numpy(),
            "route": trace.route.reshape(batch, angle_count, -1).cpu().numpy(),
            "z_mm": trace.z_mm.reshape(batch, angle_count, -1).cpu().numpy(),
            "logits": logits.cpu().numpy(),
        }
        log_probs = F.log_softmax(logits, dim=-1).cpu().numpy()
        for source in range(batch):
            label = int(labels[source])
            for angle_index in range(angle_count):
                rows.append({
                    "model_seed": model_seed, "method": method,
                    "source_index": int(source_ids[source]), "label": label,
                    "angle": float(factors[source, angle_index]),
                    "correct": int(arrays["logits"][source, angle_index].argmax() == label),
                    "true_log_prob": float(log_probs[source, angle_index, label]),
                    "context": arrays["context"][source, angle_index],
                    "context_raw": arrays["context_raw"][source, angle_index],
                    "route": arrays["route"][source, angle_index],
                    "z_mm": arrays["z_mm"][source, angle_index],
                })
    return pd.DataFrame(rows)

def stack(frame, column):
    return np.stack(frame[column].to_numpy())

def grouped_space(frame, angles, column, metric):
    sub = frame[frame.angle.isin(angles)]
    return grouped_geometry_scores(
        sub.angle.to_numpy(float), stack(sub, column), sub.source_index.to_numpy(), metric=metric
    )

def bootstrap_delta(values, samples, seed):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = np.empty(samples, dtype=float)
    for index in range(samples):
        draws[index] = values[rng.integers(0, len(values), len(values))].mean()
    return float(values.mean()), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))

def paired_geometry_delta(trace, treatment, control, angles, space):
    column = {"context": "context", "route": "route", "synthesis": "z_mm"}[space]
    metric = "fisher_rao" if space == "route" else "euclidean"
    t = grouped_space(trace[trace.method == treatment], angles, column, metric)
    c = grouped_space(trace[trace.method == control], angles, column, metric)
    rows = []
    for metric_name in ["rho", "stress"]:
        tm = dict(zip(t["group_ids"], t[metric_name])); cm = dict(zip(c["group_ids"], c[metric_name]))
        common = sorted(set(tm) & set(cm))
        delta = np.asarray([tm[g] - cm[g] for g in common])
        mean, low, high = bootstrap_delta(delta, SOURCE_BOOTSTRAP_SAMPLES, SPLIT_SEED + 19)
        rows.append({"space": space, "metric": metric_name, "delta_mean": mean, "delta_ci_low": low, "delta_ci_high": high, "source_blocks": len(common)})
    return rows

def paired_prediction_deltas(trace, treatment, control):
    rows = []
    for angle in DENSE_HELDOUT_ANGLES:
        t = trace[(trace.method == treatment) & (trace.angle == angle)][["source_index", "correct", "true_log_prob"]]
        c = trace[(trace.method == control) & (trace.angle == angle)][["source_index", "correct", "true_log_prob"]]
        merged = t.merge(c, on="source_index", suffixes=("_t", "_c"))
        for metric, values in {
            "accuracy": merged.correct_t.to_numpy(float) - merged.correct_c.to_numpy(float),
            "true_log_prob": merged.true_log_prob_t.to_numpy(float) - merged.true_log_prob_c.to_numpy(float),
        }.items():
            mean, low, high = bootstrap_delta(values, SOURCE_BOOTSTRAP_SAMPLES, SPLIT_SEED + int(angle * 10) + 101)
            rows.append({"angle": angle, "metric": metric, "delta_mean": mean, "delta_ci_low": low, "delta_ci_high": high, "source_blocks": len(merged)})
    return rows

def orthogonal_directions(tangent, count, seed):
    rng = np.random.default_rng(seed); output = []
    while len(output) < count:
        vector = rng.normal(size=tangent.shape)
        vector -= np.dot(vector, tangent) * tangent
        norm = np.linalg.norm(vector)
        if norm > 1e-12: output.append(vector / norm)
    return output

def source_bootstrap_ratio(frame, samples, seed):
    rng = np.random.default_rng(seed); n = len(frame); values = []
    for _ in range(samples):
        sample = frame.iloc[rng.integers(0, n, n)]
        values.append(abs(sample.signed_route_effect.mean()) / max(sample.orthogonal_abs_effect.mean(), 1e-12))
    point = abs(frame.signed_route_effect.mean()) / max(frame.orthogonal_abs_effect.mean(), 1e-12)
    return point, float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))

@torch.no_grad()
def causal_probe(model, method, model_seed, fit_trace, eval_indices):
    fit = fit_trace[(fit_trace.method == method) & (fit_trace.angle.isin(train_angles))]
    X = stack(fit, "context").astype(float); y = fit.angle.to_numpy(float)
    X0 = X - X.mean(0); y0 = y - y.mean()
    ridge = 1e-6 * max(np.trace(X0.T @ X0) / max(X0.shape[1], 1), 1.0)
    tangent = np.linalg.solve(X0.T @ X0 + ridge * np.eye(X0.shape[1]), X0.T @ y0)
    tangent /= max(np.linalg.norm(tangent), 1e-12)

    route_centroids = {
        float(angle): np.stack(group.route.to_numpy()).mean(0)
        for angle, group in fit.groupby("angle")
    }
    roots = np.stack([np.sqrt(np.clip(route_centroids[a], 1e-12, None)) for a in train_angles])
    centered = np.asarray(train_angles) - np.mean(train_angles)
    route_direction = (centered[:, None] * (roots - roots.mean(0))).sum(0) / max(np.sum(centered ** 2), 1e-12)
    route_direction /= max(np.linalg.norm(route_direction), 1e-12)
    orthogonals = orthogonal_directions(tangent, int(protocol["causal_protocol"]["orthogonal_controls"]), model_seed + 901)

    raw_rows = []
    probe_angle = float(protocol["causal_protocol"]["probe_angle"])
    for x, labels, _, source_ids in fixed_loader(probe_angle, False, eval_indices, False, batch_size=128):
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        base = forward_mode(model, x)
        base_logp = F.log_softmax(base.logits, -1).gather(1, labels[:, None]).squeeze(1)
        base_pred = base.logits.argmax(1)
        tangent_tensor = torch.tensor(tangent, dtype=base.context.dtype, device=DEVICE)
        route_tensor = torch.tensor(route_direction, dtype=base.route.dtype, device=DEVICE)
        for scale in protocol["causal_protocol"]["scales"]:
            projected = tangent_tensor - (base.context * tangent_tensor).sum(-1, keepdim=True) * base.context
            projected = F.normalize(projected, dim=-1)
            new_context = F.normalize(base.context + float(scale) * projected, dim=-1)
            new_route = torch.softmax(model.context_bottleneck_router(new_context), -1)
            delta_root = torch.sqrt(new_route.clamp_min(1e-12)) - torch.sqrt(base.route.clamp_min(1e-12))
            signed = (delta_root * route_tensor).sum(1)
            z = torch.einsum("bh,bhd->bd", new_route, base.host_outputs)
            new_logits = model.classifier(model.synthesis_norm(z))
            new_logp = F.log_softmax(new_logits, -1).gather(1, labels[:, None]).squeeze(1)
            new_pred = new_logits.argmax(1)

            control_effects = []
            for direction in orthogonals:
                vector = torch.tensor(direction, dtype=base.context.dtype, device=DEVICE)
                local = vector - (base.context * vector).sum(-1, keepdim=True) * base.context
                local = F.normalize(local, dim=-1)
                control_context = F.normalize(base.context + float(scale) * local, dim=-1)
                route = torch.softmax(model.context_bottleneck_router(control_context), -1)
                route_delta = torch.sqrt(route.clamp_min(1e-12)) - torch.sqrt(base.route.clamp_min(1e-12))
                control_effects.append(torch.abs((route_delta * route_tensor).sum(1)))
            orthogonal = torch.stack(control_effects).mean(0)
            for index, source_id in enumerate(source_ids):
                raw_rows.append({
                    "model_seed": model_seed, "method": method, "source_index": int(source_id),
                    "scale": float(scale), "signed_route_effect": float(signed[index].cpu()),
                    "orthogonal_abs_effect": float(orthogonal[index].cpu()),
                    "class_log_prob_drop": float((base_logp[index] - new_logp[index]).cpu()),
                    "prediction_preserved": int(base_pred[index] == new_pred[index]),
                })
    raw = pd.DataFrame(raw_rows)
    summary_rows = []
    for scale, block in raw.groupby("scale"):
        if scale == 0:
            csr, csr_low, csr_high = 0.0, 0.0, 0.0
        else:
            csr, csr_low, csr_high = source_bootstrap_ratio(
                block, int(protocol["causal_protocol"]["source_bootstrap_samples"]), model_seed + int(abs(scale) * 1000) + 77
            )
        preservation = block.prediction_preserved.to_numpy(float)
        p_mean, p_low, p_high = bootstrap_delta(preservation, SOURCE_BOOTSTRAP_SAMPLES, model_seed + 404)
        summary_rows.append({
            "model_seed": model_seed, "method": method, "scale": scale,
            "csr": csr, "csr_ci_low": csr_low, "csr_ci_high": csr_high,
            "signed_route_effect": block.signed_route_effect.mean(),
            "orthogonal_abs_effect": block.orthogonal_abs_effect.mean(),
            "class_log_prob_drop": block.class_log_prob_drop.mean(),
            "prediction_preservation": p_mean,
            "prediction_preservation_ci_low": p_low,
            "prediction_preservation_ci_high": p_high,
        })
    return raw, pd.DataFrame(summary_rows)


## 6. Execute each model seed


In [ ]:
def run_seed(model_seed):
    print("\n===== MODEL SEED", model_seed, "=====")
    random.seed(model_seed); np.random.seed(model_seed); torch.manual_seed(model_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(model_seed)
    torch.manual_seed(model_seed + 1090)
    base_model = build_model(); initial_state = copy.deepcopy(base_model.state_dict())
    initial_hash = state_hash(initial_state); del base_model

    models = {}; stage_tables = []; evaluation_tables = []
    for method in METHODS:
        print("\n---", method, "---")
        model, stage_table, evaluation_table = train_method(model_seed, method, METHOD_SPECS[method], initial_state)
        models[method] = model; stage_tables.append(stage_table); evaluation_tables.append(evaluation_table)
    stage = pd.concat(stage_tables, ignore_index=True)
    evaluation = pd.concat(evaluation_tables, ignore_index=True)

    # Hard equal-compute gate within seed.
    for stage_id in sorted(stage.stage.unique()):
        block = stage[stage.stage == stage_id]
        assert block.forward_images.nunique() == 1 and block.optimizer_steps.nunique() == 1

    trace = pd.concat(
        [collect_trace(models[m], m, model_seed, DENSE_EVAL_ANGLES, test_indices) for m in METHODS],
        ignore_index=True,
    )
    fit_trace = pd.concat(
        [collect_trace(models[m], m, model_seed, train_angles, tangent_fit_indices) for m in METHODS],
        ignore_index=True,
    )

    method_rows = []
    forgetting_rows = []
    geometry_rows = []
    factor_rows = []
    causal_raw_tables = []; causal_summary_tables = []
    final_stage = int(evaluation.stage.max())
    for method in METHODS:
        method_trace = trace[trace.method == method]
        # Geometry partitions.
        values = {}
        for partition, angles in {"C1_trained": train_angles, "dense_heldout": DENSE_HELDOUT_ANGLES}.items():
            for space, column, metric in [
                ("context", "context", "euclidean"),
                ("route", "route", "fisher_rao"),
                ("synthesis", "z_mm", "euclidean"),
            ]:
                grouped = grouped_space(method_trace, angles, column, metric)
                rho_mean, rho_low, rho_high = bootstrap_mean_ci(grouped["rho"], SOURCE_BOOTSTRAP_SAMPLES, seed=model_seed + 11)
                stress_mean, stress_low, stress_high = bootstrap_mean_ci(grouped["stress"], SOURCE_BOOTSTRAP_SAMPLES, seed=model_seed + 12)
                centroid = centroid_geometry_scores(
                    method_trace[method_trace.angle.isin(angles)].angle.to_numpy(float),
                    stack(method_trace[method_trace.angle.isin(angles)], column),
                    metric=metric,
                )
                geometry_rows.append({
                    "model_seed": model_seed, "method": method, "partition": partition, "space": space,
                    "source_rho_mean": rho_mean, "source_rho_ci_low": rho_low, "source_rho_ci_high": rho_high,
                    "source_stress_mean": stress_mean, "source_stress_ci_low": stress_low, "source_stress_ci_high": stress_high,
                    "centroid_rho": centroid["rho"], "centroid_stress": centroid["stress"],
                })
                values[(partition, space, "rho")] = rho_mean
                values[(partition, space, "stress")] = stress_mean

        fit = fit_trace[fit_trace.method == method]
        heldout = method_trace[method_trace.angle.isin(DENSE_HELDOUT_ANGLES) & method_trace.source_index.isin(tangent_eval_indices)]
        for space in ["context", "route"]:
            probe = ridge_factor_probe(
                stack(fit, space), fit.angle.to_numpy(float),
                stack(heldout, space), heldout.angle.to_numpy(float),
            )
            factor_rows.append({"model_seed": model_seed, "method": method, "space": space, **probe})

        # Forgetting and final accuracy.
        method_eval = evaluation[evaluation.method == method]
        for angle in train_angles:
            seen_stage = PRIMARY_CURRICULUM.index(angle)
            rows = method_eval[(method_eval.angle == angle) & (method_eval.stage >= seen_stage)]
            best = float(rows.accuracy.max()); final_acc = float(rows[rows.stage == final_stage].accuracy.iloc[0])
            forgetting_rows.append({"model_seed": model_seed, "method": method, "angle": angle, "best_accuracy": best, "final_accuracy": final_acc, "forgetting": best - final_acc})
        final = method_eval[method_eval.stage == final_stage]
        method_rows.append({
            "model_seed": model_seed, "method": method,
            "context_rho": values[("C1_trained", "context", "rho")],
            "route_rho": values[("C1_trained", "route", "rho")],
            "synthesis_rho": values[("C1_trained", "synthesis", "rho")],
            "dense_context_rho": values[("dense_heldout", "context", "rho")],
            "dense_route_rho": values[("dense_heldout", "route", "rho")],
            "mean_heldout_accuracy": final[final.angle.isin(DENSE_HELDOUT_ANGLES)].accuracy.mean(),
            "final_trained_accuracy": final[final.angle.isin(train_angles)].accuracy.mean(),
        })

        causal_raw, causal_summary = causal_probe(models[method], method, model_seed, fit_trace, tangent_eval_indices)
        causal_raw_tables.append(causal_raw); causal_summary_tables.append(causal_summary)

    geometry = pd.DataFrame(geometry_rows)
    method_summary = pd.DataFrame(method_rows)
    forgetting = pd.DataFrame(forgetting_rows)
    method_summary = method_summary.merge(
        forgetting.groupby(["model_seed", "method"], as_index=False).forgetting.mean().rename(columns={"forgetting": "mean_forgetting"}),
        on=["model_seed", "method"], how="left",
    )
    factor_probe = pd.DataFrame(factor_rows)
    method_summary = method_summary.merge(
        factor_probe[factor_probe.space == "context"][["model_seed", "method", "r2", "mae"]].rename(columns={"r2": "heldout_context_r2", "mae": "heldout_context_mae"}),
        on=["model_seed", "method"], how="left",
    )

    # Paired source effects and prediction effects.
    contrast_rows = []; prediction_rows = []
    for contrast, treatment, control in [
        ("stationary_full_vs_no_geo", "d4_full_stationary_route", "d4_no_geo"),
        ("stationary_vs_legacy_route", "d4_full_stationary_route", "d4_full_legacy_route"),
    ]:
        for partition, angles in {"C1_trained": train_angles, "dense_heldout": DENSE_HELDOUT_ANGLES}.items():
            for space in ["context", "route", "synthesis"]:
                for row in paired_geometry_delta(trace, treatment, control, angles, space):
                    contrast_rows.append({"model_seed": model_seed, "contrast": contrast, "partition": partition, **row})
        for row in paired_prediction_deltas(trace, treatment, control):
            prediction_rows.append({"model_seed": model_seed, "contrast": contrast, **row})

    return {
        "initial_hash": initial_hash,
        "stage": stage,
        "evaluation": evaluation,
        "trace": trace,
        "geometry": geometry,
        "factor_probe": factor_probe,
        "forgetting": forgetting,
        "method_summary": method_summary,
        "contrast": pd.DataFrame(contrast_rows),
        "prediction": pd.DataFrame(prediction_rows),
        "causal_raw": pd.concat(causal_raw_tables, ignore_index=True),
        "causal_summary": pd.concat(causal_summary_tables, ignore_index=True),
    }

all_runs = []
for model_seed in MODEL_SEEDS:
    all_runs.append((model_seed, run_seed(model_seed)))
print("Completed seeds:", [seed for seed, _ in all_runs])


## 7. Aggregate model-seed evidence


In [ ]:
def concat_key(key):
    return pd.concat([run[key] for _, run in all_runs], ignore_index=True)

stage_df = concat_key("stage")
evaluation_df = concat_key("evaluation")
geometry_df = concat_key("geometry")
factor_probe_df = concat_key("factor_probe")
forgetting_df = concat_key("forgetting")
per_seed_method_df = concat_key("method_summary")
per_seed_contrast_df = concat_key("contrast")
paired_prediction_df = concat_key("prediction")
causal_raw_df = concat_key("causal_raw")
causal_summary_df = concat_key("causal_summary")
initial_state_hashes = {str(seed): run["initial_hash"] for seed, run in all_runs}

compute_summary_df = stage_df.groupby(["model_seed", "method"], as_index=False).agg(
    total_forward_images=("forward_images", "sum"),
    total_optimizer_steps=("optimizer_steps", "sum"),
    total_wall_seconds=("wall_seconds", "sum"),
)

def seed_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float); n = len(values)
    mean = float(values.mean()); sd = float(values.std(ddof=1)) if n > 1 else 0.0
    if n < 2: return mean, mean, mean, sd, n
    half = float(stats.t.ppf((1 + confidence) / 2, df=n - 1) * sd / math.sqrt(n))
    return mean, mean - half, mean + half, sd, n

aggregate_rows = []
metric_columns = [
    "context_rho", "route_rho", "synthesis_rho", "dense_context_rho", "dense_route_rho",
    "heldout_context_r2", "heldout_context_mae", "mean_heldout_accuracy",
    "final_trained_accuracy", "mean_forgetting",
]
for method, block in per_seed_method_df.groupby("method"):
    for metric in metric_columns:
        mean, low, high, sd, n = seed_ci(block[metric])
        aggregate_rows.append({"method": method, "metric": metric, "mean": mean, "seed_ci_low": low, "seed_ci_high": high, "seed_sd": sd, "seed_count": n})
aggregate_method_df = pd.DataFrame(aggregate_rows)

# Seed-level treatment effects use the per-seed source-bootstrap point estimates.
seed_effect_rows = []
for keys, block in per_seed_contrast_df.groupby(["contrast", "partition", "space", "metric"]):
    mean, low, high, sd, n = seed_ci(block.delta_mean)
    seed_effect_rows.append({
        "contrast": keys[0], "partition": keys[1], "space": keys[2], "metric": keys[3],
        "mean_seed_effect": mean, "seed_ci_low": low, "seed_ci_high": high,
        "seed_sd": sd, "seed_count": n,
        "positive_seed_fraction": float(np.mean(block.delta_mean > 0)),
    })
aggregate_effect_df = pd.DataFrame(seed_effect_rows)

# Aggregate nonzero-scale causal candidate evidence by seed and method.
causal_seed_df = (
    causal_summary_df[causal_summary_df.scale != 0]
    .groupby(["model_seed", "method"], as_index=False)
    .agg(
        mean_csr=("csr", "mean"),
        min_source_csr_ci_low=("csr_ci_low", "min"),
        mean_prediction_preservation=("prediction_preservation", "mean"),
        min_prediction_preservation_ci_low=("prediction_preservation_ci_low", "min"),
        mean_class_log_prob_drop=("class_log_prob_drop", "mean"),
    )
)
causal_aggregate_rows = []
for method, block in causal_seed_df.groupby("method"):
    for metric in ["mean_csr", "min_source_csr_ci_low", "mean_prediction_preservation", "min_prediction_preservation_ci_low", "mean_class_log_prob_drop"]:
        mean, low, high, sd, n = seed_ci(block[metric])
        causal_aggregate_rows.append({"method": method, "metric": metric, "mean": mean, "seed_ci_low": low, "seed_ci_high": high, "seed_sd": sd, "seed_count": n})
causal_aggregate_df = pd.DataFrame(causal_aggregate_rows)

aggregate_method_df.head(), aggregate_effect_df.head(), causal_aggregate_df.head()


## 8. Pilot gates and epistemic status


In [ ]:
def aggregate_effect(contrast, partition, space, metric):
    row = aggregate_effect_df[
        (aggregate_effect_df.contrast == contrast)
        & (aggregate_effect_df.partition == partition)
        & (aggregate_effect_df.space == space)
        & (aggregate_effect_df.metric == metric)
    ].iloc[0]
    return row

route_effect = aggregate_effect("stationary_full_vs_no_geo", "C1_trained", "route", "rho")
context_effect = aggregate_effect("stationary_full_vs_no_geo", "C1_trained", "context", "rho")
legacy_effect = aggregate_effect("stationary_vs_legacy_route", "C1_trained", "route", "rho")
stationary_causal = causal_seed_df[causal_seed_df.method == "d4_full_stationary_route"]

candidate_gates = pd.DataFrame([
    {
        "gate": "C0_equal_compute_all_seeds",
        "passed": bool(compute_summary_df.groupby("model_seed").apply(lambda b: b.total_forward_images.nunique() == 1 and b.total_optimizer_steps.nunique() == 1).all()),
        "status": "pilot_protocol",
    },
    {
        "gate": "C1_stationary_route_rho_seed_ci",
        "passed": bool(route_effect.seed_ci_low > float(protocol["pilot_gates"]["route_rho_seed_ci_low"])),
        "status": "candidate_only",
    },
    {
        "gate": "C1_stationary_context_rho_seed_ci",
        "passed": bool(context_effect.seed_ci_low > float(protocol["pilot_gates"]["context_rho_seed_ci_low"])),
        "status": "candidate_only",
    },
    {
        "gate": "stationary_beats_legacy_route_target",
        "passed": bool(legacy_effect.seed_ci_low > 0),
        "status": "diagnostic",
    },
    {
        "gate": "C4_source_bootstrap_csr",
        "passed": bool((stationary_causal.min_source_csr_ci_low > float(protocol["pilot_gates"]["causal_csr_source_ci_low"])).mean() >= float(protocol["pilot_gates"]["minimum_positive_seed_fraction"])),
        "status": "candidate_only",
    },
    {
        "gate": "C4_identity_preservation",
        "passed": bool((stationary_causal.min_prediction_preservation_ci_low > float(protocol["pilot_gates"]["identity_preservation_source_ci_low"])).mean() >= float(protocol["pilot_gates"]["minimum_positive_seed_fraction"])),
        "status": "candidate_only",
    },
    {"gate": "C2_predictive_superiority", "passed": False, "status": "requires_seed_and_source_intervals"},
    {"gate": "C3_host_specialization", "passed": False, "status": "not_evaluated"},
    {"gate": "C5_memory_transport", "passed": False, "status": "not_implemented"},
    {"gate": "C6_operational_utility", "passed": False, "status": "not_qualified"},
])
candidate_gates


## 9. Export per-seed and aggregate evidence


In [ ]:
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

root_out = Path("results/v1_0_9")
root_out.mkdir(parents=True, exist_ok=True)
for model_seed, run in all_runs:
    seed_out = root_out / f"seed_{model_seed}"
    seed_out.mkdir(parents=True, exist_ok=True)
    for filename, key in {
        "stage_logs.csv": "stage",
        "staged_evaluation.csv": "evaluation",
        "dense_trace.pkl": "trace",
        "geometry_metrics.csv": "geometry",
        "factor_probe.csv": "factor_probe",
        "forgetting.csv": "forgetting",
        "method_summary.csv": "method_summary",
        "paired_source_effects.csv": "contrast",
        "paired_prediction_effects.csv": "prediction",
        "causal_probe_raw.csv": "causal_raw",
        "causal_probe_summary.csv": "causal_summary",
    }.items():
        value = run[key]
        if filename.endswith(".pkl"):
            value.to_pickle(seed_out / filename)
        else:
            value.to_csv(seed_out / filename, index=False)

for filename, frame in {
    "per_seed_method_summary.csv": per_seed_method_df,
    "aggregate_method_summary.csv": aggregate_method_df,
    "per_seed_source_effects.csv": per_seed_contrast_df,
    "aggregate_seed_effects.csv": aggregate_effect_df,
    "per_seed_prediction_effects.csv": paired_prediction_df,
    "causal_probe_source_summary.csv": causal_summary_df,
    "causal_seed_summary.csv": causal_seed_df,
    "causal_aggregate_summary.csv": causal_aggregate_df,
    "compute_summary.csv": compute_summary_df,
    "pilot_gate_summary.csv": candidate_gates,
    "sensory_history.csv": pd.DataFrame(sensory_history),
}.items():
    frame.to_csv(root_out / filename, index=False)

manifest = {
    "status": "pilot_not_final_qualification",
    "notebook_version": NOTEBOOK_VERSION,
    "build_revision": BUILD_REVISION,
    "package_version": geometry_mmalls.__version__,
    "git_sha": git_sha,
    "run_profile": RUN_PROFILE,
    "model_seeds": MODEL_SEEDS,
    "fixed_split_seed": SPLIT_SEED,
    "fixed_sensory_seed": SENSORY_SEED,
    "initial_state_hashes": initial_state_hashes,
    "split_manifest": split_manifest,
    "sensory_accuracy": sensory_accuracy,
    "method_specs": METHOD_SPECS,
    "stationary_route_geometry": protocol["route_geometry"],
    "dense_eval_angles": DENSE_EVAL_ANGLES,
    "tracked_changes": [f"CHG-109-{index:02d}" for index in range(1, 15)],
    "archived_v108_results": "results/v1_0_8/seed_0",
    "v108_results_report": "docs/reports/Geometry_MMALS_G1_v1_0_8_Results_and_Interpretation_Report.pdf",
    "reviewer_report": "docs/reports/Geometry_MMALS_G1_Status_and_Perspective_Reviewer_Report_v1_1.pdf",
    "warning": "Five-seed pilot evidence is not final C1-C6 qualification.",
    "timestamp": time.time(),
}
(root_out / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Exported to", root_out.resolve())


## 10. Interpretation rules

A successful pilot requires more than one favorable seed. The primary evidence unit is the **model seed**, while source bootstrap intervals remain the within-seed uncertainty measure.

The stationary treatment is a candidate C1 success only if:

1. the seed-level confidence interval for trained route-order improvement is positive;
2. the seed-level confidence interval for trained context-order improvement is positive;
3. at least 80% of seeds have a positive paired source effect;
4. dense held-out geometry and factor decoding do not collapse;
5. source-bootstrap causal specificity remains above one with preserved predictions;
6. all equal-compute and release-integrity gates pass.

Even if these conditions pass, predictive and operational claims require separate evidence.


## 11. Qualification checklist

- [ ] Public repository and notebook both report v1.0.9.
- [ ] Fixed source and sensory hashes match the archived v1.0.8 boundary.
- [ ] All three methods start from the same state within each model seed.
- [ ] Forward images and optimizer steps match within every seed.
- [ ] The stationary route target is invariant to continual-stage span.
- [ ] All five pilot seeds are retained, including failed seeds.
- [ ] Source-bootstrap and model-seed intervals are both exported.
- [ ] Dense held-out angles never enter training or checkpoint selection.
- [ ] Causal tangent fitting and evaluation source identities remain disjoint.
- [ ] Prediction identity is reported under causal interventions.
- [ ] v1.0.8 and v1.0.9 reports remain explicitly non-overclaiming.
